# Trace a Video Insert with OpenTelemetry

In this video, I'm going to send Pixeltable's traces to Phoenix and follow one video insert all the way down to each OpenAI call on its frames. The trace shows where the time goes and how many tokens each call uses, and we'll use it to cut the input tokens per frame from 14,182 to 2,848.

Model calls on video frames are slow, and you pay for every token. These calls also run in parallel, off the line that called `insert`, and the trace still puts every one of them inside that insert.

This comes from our blog post on tracing multimodal AI pipelines with OpenTelemetry, which also covers four other tracing tools. We'll look at three traces: an insert, a read, and a recompute.

## Phoenix runs on your laptop and receives the traces

Phoenix is an open-source tracing tool. Off camera, I started it in a terminal with `uvx --from arize-phoenix phoenix serve`, and my OpenAI key is in `~/.pixeltable/config.toml`. The notebook is in the `otel-tracing` folder of the repo linked below.

In [ ]:
!uv sync

## Tracing stays off until you give it an endpoint

Tracing is off by default. `pxt_otel.init` turns it on and sends every Pixeltable operation to the endpoint you give it, which is Phoenix on port 6006. Call it once, before any table operation.

In [ ]:
import opentelemetry.instrumentation.pixeltable as pxt_otel
from openinference.instrumentation.openai import OpenAIInstrumentor

pxt_otel.init(endpoint='http://localhost:6006', span_level='trace', metrics=False)
OpenAIInstrumentor().instrument()

`span_level='trace'` adds a span for every frame, which is fine for a short video. On a long one, use `debug`. Phoenix takes traces only, so we turn metrics off.

Pixeltable's own spans record what ran and how long it took. The OpenAI instrumentor adds each model call's prompt and answer, and those go to the endpoint too. With a hosted tool, your prompts leave your laptop.

In [ ]:
import pixeltable as pxt

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('otel_demo', force=True, if_not_exists='ignore')

Every Pixeltable operation is its own trace, so even this reset shows up in Phoenix.

## Each video gets one row per frame, and each frame gets a description

In `videos`, the only column you fill in is `video`. `frames` is a view with one row per frame, at one frame every two seconds.

In [ ]:
!sed -n '11,20p' schema.py

Each frame gets a computed column, `response`, that calls gpt-4o-mini. The prompt is this one sentence plus the image from the `frame` column. `description` pulls the text out of the response.

In [ ]:
!sed -n '8p;21,33p' schema.py

`pxt schema diff` shows the plan, and `pxt schema update` creates the tables. These two commands run outside this notebook, so they don't show up in Phoenix.

In [ ]:
!pxt schema diff schema.py otel_demo

In [ ]:
!pxt schema update schema.py otel_demo -f

## One insert is one trace, from the download to the last model call

I'll insert one 18-second street video from the Pixeltable repo. Every frame gets its description inside this one insert.

In [ ]:
BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/tests/data/videos/'

videos = pxt.get_table('otel_demo.videos')
videos.insert([{'video': BASE + 'bangkok_half_res.mp4'}])

That's 11 rows: the video, and its 10 frames.

*Switch to Phoenix at localhost:6006, open the `default` project, and click the `pixeltable.insert` trace.*

Here's the whole insert as one trace. `pixeltable.media.fetch` is the download, and each `frame_iterator` span is one frame.

Further down, each `chat_completions` span is one call to OpenAI. Most of the five seconds is these calls, and Phoenix prices the whole insert at about two cents.

The first call runs alone, because Pixeltable reads OpenAI's rate limits from its response. Then the other nine run at once.

Those calls ran on async tasks, not on the line that called `insert`, and each one still lands in this trace.

## Each model call carries its prompt and its tokens

*Click a `ChatCompletion` span.*

The OpenAI instrumentor adds this span. It shows the prompt this frame got: our sentence, then the image. The image is redacted, because the instrumentor leaves out large base64 images.

*Hover over the token count.*

This call used 14,182 input tokens and about 30 output tokens. Almost all of the input is the image.

## Reading the descriptions back calls no model

The descriptions are stored in the table, next to each frame.

In [ ]:
frames = pxt.get_table('otel_demo.frames')
frames.select(frames.frame, frames.description).order_by(frames.pos).collect()

*Switch to Phoenix and open the newest `pixeltable.collect` trace.*

Reading them back is its own trace, `pixeltable.collect`, and it has no OpenAI span.

## A low-detail image cuts the input tokens to 2,848

OpenAI's `detail: 'low'` option sends a smaller version of the image. It's a one-line change in the schema.

In [ ]:
!diff schema.py schema_low.py

In [ ]:
!pxt schema diff schema_low.py otel_demo

The value expression of `response` will be updated, and existing values won't be recomputed. It's marked safe.

In [ ]:
!pxt schema update schema_low.py otel_demo -f

I want every frame on the same setting, so I'll recompute `response` on purpose.

In [ ]:
frames = pxt.get_table('otel_demo.frames')  # reload after the schema change
frames.recompute_columns('response').num_rows

*Switch to Phoenix, open the `pixeltable.recompute_columns` trace, and hover over a `ChatCompletion` token count.*

The recompute is one more trace, with 10 calls. Each one used 2,848 input tokens, about a fifth of what it used before.

In [ ]:
frames.select(frames.frame, frames.description).order_by(frames.pos).collect()

The descriptions still pick out the traffic and the buildings. Check them on your own video before you settle on low detail.

## Three traces showed the time, the tokens, and the fix

We traced one insert, from the download to each model call. Each call showed its tokens. After a one-line change, the recompute's trace showed each frame drop from 14,182 input tokens to 2,848.

To send the same traces to Langfuse or MLflow, you change the endpoint and the headers. The README lists both for each tool.

The notebook and schema files are in the `otel-tracing` folder of the repo linked below. If this helps, a star on the Pixeltable GitHub repo helps other developers find it.